# Biological Concordance Methodology

## 1. Introduction: The Trimming Paradox in RNA-Seq
Standard RNA-Seq pipelines universally employ read trimming to remove adapters and low-quality bases, operating under the assumption that aggressive quality control uniformly improves downstream biological signal. However, modern aligners soft-clip effectively, raising the question of whether aggressive trimming introduces artificial variance and destroys biologically relevant information. 

To evaluate this systematically, we developed a robust, large-scale **Leave-One-Sample-Out (LOSO) Cross-Validation Framework** across dozens of diverse BioProjects to quantify the exact impact of sequence trimming on downstream biological concordance.

## 2. Experimental Design: Leave-One-Sample-Out (LOSO)

To definitively prove whether a specific trimming parameter introduces variance, we must mathematically isolate the target sample from the background population. 

If a BioProject contains $N$ samples, our LOSO framework executes the following for each sample $i$:
1. The **Target Sample ($i$)** is processed using the candidate trimming algorithm being evaluated (e.g., Un-trimmed, Adaptive, Phred-5, Phred-10, Phred-20, Phred-35).
2. The remaining **Background Population ($N-1$ samples)** is uniformly processed using a strictly controlled **Project-Baseline Trimming Method**.
3. All $N$ samples are combined into a single count matrix, and differential expression is computed using PyDESeq2 against the project's biological conditions.
4. This is repeated iteratively for all candidate trimming methods, and all samples.

## 3. Anticipating Reviewer Feedback: The "Baseline Anchor" Defense

A critical methodological decision in our LOSO framework is the use of a single, optimized **Project-Baseline** for the $N-1$ background population, rather than testing an all-vs-all combinatorial matrix of trimming methods. This design is highly intentional and serves three critical mathematical purposes:

### A. Statistical Isolation (The "Stable Anchor" Principle)
PyDESeq2 computes empirical $p$-values by modeling the dispersion of the entire population. To measure variance injected *strictly* by altering the target sample's trimming method, the background population variance must be completely frozen. If the background trimming methods were varied simultaneously with the target method, it would introduce confounding variables—making it mathematically impossible to determine whether a shift in biological concordance originated from the target sample or a shift in the background dispersion model.

### B. Maximizing Statistical Power
The Project-Baseline method is not chosen arbitrarily. Our pipeline dynamically scans the alignment rates for each BioProject and automatically selects the least aggressive trimming method that successfully yields the highest fraction of mapped reads. By anchoring the $N-1$ background to the method with the highest read retention, we mathematically guarantee that the baseline population has the deepest possible sequencing coverage and the lowest technical noise. This provides PyDESeq2 with maximum statistical power to detect subtle variance in the target sample. Anchoring to highly aggressive, read-destructive methods (like Phred-35) would artificially inflate the background noise, masking the biological signal.

### C. Combinatorial Feasibility
Executing an all-vs-all comparison (testing 6 target methods against 6 background methods) would require generating $6 \times 6 = 36$ differential expression models per sample. For an 800-sample dataset, this requires over 28,800 PyDESeq2 runs. Our framework isolates the exact same variance using a statistically superior fixed-anchor model in only 4,800 runs, optimizing computational resources without sacrificing statistical integrity.

## 4. Evaluation Metrics

To quantify the stability of the biological signal across trimming methods, we evaluate concordance at three levels:

1. **Gene-Level Log2 Fold-Change (Spearman's Rho)**: Measures the global rank-order stability of fold changes across all expressed genes.
2. **Differentially Expressed Gene (DEG) Concordance (Jaccard Index)**: Quantifies the absolute overlap of statistically significant DEGs (adjusted $p < 0.05$).
3. **Pathway-Level Concordance (GSEA)**: Measures the stability of higher-order biological interpretations by evaluating Jaccard overlap of enriched GO/KEGG pathways.

## 5. Detailed Pipeline & Parameter Configuration

To ensure reproducibility and strict mathematical control, we explicitly define the core parameters used for differential expression and pathway enrichment. We deviate from default tool settings in specific areas to optimize for the LOSO cross-validation design.

### A. Differential Expression (PyDESeq2)
We employ `PyDESeq2` for all differential expression calculations. The count matrix is constructed as an $N \times G$ matrix (samples $\times$ genes).
- **Design Factor**: We use a single-factor design (`~ condition`), extracting the biological conditions directly from the standardized metadata `sample_sheets`.
- **Reference Level**: The control/baseline condition is automatically inferred as the first condition alphabetically to ensure consistency.
- **Cook's Distance Refitting**: We explicitly enable Cook's distance outlier refitting (`refit_cooks=True`) to automatically handle extreme expression outliers caused by technical sequencing artifacts.
- **Independent Filtering**: By default, PyDESeq2 performs independent filtering to optimize power. We do not disable this, as we want the true biological signal to dictate statistical power.

### B. Gene Set Enrichment Analysis (GSEA)
To evaluate pathway concordance, we perform pre-ranked GSEA using `gseapy.prerank` against standard MSigDB pathway databases (e.g., KEGG/GO).
- **Ranking Metric**: Genes are strictly ranked by their PyDESeq2 **Wald Statistic** (`stat`). This provides a continuous, signed metric that perfectly captures both the magnitude (fold change) and the significance ($p$-value) of the shift. Any genes with a Wald statistic of exactly `0.0` are excluded from the ranking.
- **Set Size Constraints**: We restrict pathway sizes to `min_size=15` and `max_size=500` to avoid overly broad or overly specific biological terms.
- **Permutations**: To optimize computational performance across 4,800+ models, we use `permutation_num=100`. While lower than the standard 1000, it is sufficient to capture the macroscopic pathway-level Jaccard overlaps.
- **Seed**: A fixed `seed=42` is explicitly set to guarantee deterministic GSEA output across runs, eliminating random permutation variance from our concordance calculations.